In [1]:
from google.colab import drive
import os
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import zipfile

# 2. Define paths
# 'My Drive' is the standard root folder of your Google Drive
zip_path = '/content/drive/MyDrive/IML/EX4/whichfaceisreal.zip'
extract_path = '/content/dataset/whichfaceisrael'  # We will extract it here to keep it organized

# 3. Unzip the file
print(f"Extracting {zip_path}...")

# Check if the file exists first to avoid errors
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Done! Files extracted to:", extract_path)
else:
    print(f"Error: File not found at {zip_path}. Please check the filename and path.")

Extracting /content/drive/MyDrive/IML/EX4/whichfaceisreal.zip...
Done! Files extracted to: /content/dataset/whichfaceisrael


In [6]:
import os
import torch
import torchvision
from torchvision import transforms
from tqdm import tqdm
import numpy as np
from xgboost import XGBClassifier

# Set seed
np.random.seed(0)

def get_loaders(path, transform, batch_size):
    train_set = torchvision.datasets.ImageFolder(root=os.path.join(path, 'train'), transform=transform)
    val_set = torchvision.datasets.ImageFolder(root=os.path.join(path, 'val'), transform=transform)
    test_set = torchvision.datasets.ImageFolder(root=os.path.join(path, 'test'), transform=transform)

    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.CenterCrop(64), transforms.ToTensor()])
batch_size = 32
path = '/content/dataset/whichfaceisreal' # UPDATE THIS PATH

train_loader, val_loader, test_loader = get_loaders(path, transform, batch_size)

# DATA LOADING (Flattening images for XGBoost)
train_data = []
train_labels = []
test_data = []
test_labels = []

print("Loading and flattening Training data...")
with torch.no_grad():
    for (imgs, labels) in tqdm(train_loader, total=len(train_loader), desc='Train'):
        train_data.append(imgs)
        train_labels.append(labels)
    train_data = torch.cat(train_data, 0).cpu().numpy().reshape(len(train_loader.dataset), -1)
    train_labels = torch.cat(train_labels, 0).cpu().numpy()

print("Loading and flattening Test data...")
with torch.no_grad():
    for (imgs, labels) in tqdm(test_loader, total=len(test_loader), desc='Test'):
        test_data.append(imgs)
        test_labels.append(labels)
    test_data = torch.cat(test_data, 0).cpu().numpy().reshape(len(test_loader.dataset), -1)
    test_labels = torch.cat(test_labels, 0).cpu().numpy()

Loading and flattening Training data...


Train: 100%|██████████| 44/44 [01:09<00:00,  1.58s/it]


Loading and flattening Test data...


Test: 100%|██████████| 13/13 [00:17<00:00,  1.34s/it]


In [10]:
print("Training XGBoost Classifier...")

# Initialize XGBoost with default parameters
model = XGBClassifier()

# Fit the model
model.fit(train_data, train_labels)

# Evaluation
train_acc = model.score(train_data, train_labels)
test_acc = model.score(test_data, test_labels)

print(f"XGBoost Train Accuracy: {train_acc:.4f}")
print(f"XGBoost Test Accuracy: {test_acc:.4f}")

Training XGBoost Classifier...
XGBoost Train Accuracy: 1.0000
XGBoost Test Accuracy: 0.7350
